## RAG

[RAG 구현하는 방법]
1. 다양한 형식의 문서들을 텍스트로 변환한 뒤 더 작은 chunk로 분할
2. chunk들을 embedding 모델을 이용해 벡터로 변환
3. 벡터 유사도 검색을 통해 질문과 유사한 내용들을 찾음 -> @tool로 구현
4. agent에 tool로 입력

### 1. 다양한 형식의 문서들을 텍스트로 변환하기

실제 데이터들은 굉장히 다양한 형태로 존재합니다. 

word, pdf, excel 등의 파일 내용들을 텍스트로 변환해 봅시다.

#### 1-1. excel 파일 처리하기

엑셀 파일을 처리해 보겠습니다. 엑셀 파일은 아래와 같은 과정으로 데이터를 정리할 수 있습니다.

1. pandas 라이브러리를 이용해 엑셀 파일 로드
2. 엑셀 파일에 존재하는 시트 별로 데이터 처리 시작
3. 각 시트에 존재하는 행 데이터들을 텍스트로 변환하여 저장.
4. 3번 과정을 모든 시트에 대해 수행.
5. 최종적으로 모든 시트의 행 데이터들이 텍스트로 변환되어 종합됨.

In [18]:
import pandas as pd
from pathlib import Path
from langchain_core.documents import Document


def parse_excel(file_path: str) -> list[Document]:
    with pd.ExcelFile(file_path) as xl:
        docs = []
        for sheet_name in xl.sheet_names:
            row_data = ""
            df = xl.parse(sheet_name, header=1)
            for idx, row in df.iterrows():
                text = "\n".join(f"{col}: {row[col]}" for col in df.columns)
                row_data += text + "\n"
            docs.append(Document(page_content=row_data, metadata={"sheet": sheet_name}))
    return docs


excel_contents = parse_excel("data/키키테크_임직원및프로젝트현황.xlsx")
print(f"총 Document 수: {len(excel_contents)}")
print(excel_contents[0])

총 Document 수: 4
page_content='사번: TB-001
성명: 김민준
부서: AI 연구개발본부
팀: LLM팀
직급: 수석연구원
입사일: 2015-03-02
담당업무: LLM 파인튜닝 및 프롬프트 엔지니어링
연락처(내선): 5101
이메일: minjun.kim@kikitech.co.kr
근무지: 본사
비고: 팀장
사번: TB-002
성명: 이서연
부서: AI 연구개발본부
팀: LLM팀
직급: 책임연구원
입사일: 2018-07-01
담당업무: RAG 파이프라인 설계 및 최적화
연락처(내선): 5102
이메일: seoyeon.lee@kikitech.co.kr
근무지: 본사
비고: nan
사번: TB-003
성명: 박지호
부서: AI 연구개발본부
팀: LLM팀
직급: 선임연구원
입사일: 2020-09-14
담당업무: 벡터 DB 구축 및 임베딩 모델 평가
연락처(내선): 5103
이메일: jiho.park@kikitech.co.kr
근무지: 본사
비고: nan
사번: TB-004
성명: 최유진
부서: AI 연구개발본부
팀: LLM팀
직급: 연구원
입사일: 2023-02-27
담당업무: 데이터 전처리 및 평가 데이터셋 구축
연락처(내선): 5104
이메일: yujin.choi@kikitech.co.kr
근무지: 본사
비고: nan
사번: TB-005
성명: 정현우
부서: AI 연구개발본부
팀: ML팀
직급: 수석연구원
입사일: 2016-05-09
담당업무: 추천 시스템 및 분류 모델 개발
연락처(내선): 5201
이메일: hyunwoo.jung@kikitech.co.kr
근무지: 본사
비고: 팀장
사번: TB-006
성명: 한소희
부서: AI 연구개발본부
팀: ML팀
직급: 책임연구원
입사일: 2019-01-07
담당업무: 시계열 예측 모델 연구
연락처(내선): 5202
이메일: sohee.han@kikitech.co.kr
근무지: 본사
비고: nan
사번: TB-007
성명: 오지민
부서: AI 연구개발본부
팀: ML팀
직급: 선임연구원
입사일: 20

이제 이 문서 내용을 토큰 길이에 따라 분할해 보겠습니다!

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = 한 청크의 최대 토큰 수 / chunk_overlap = 겹치는 영역의 토큰 수
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
excel_chunks = splitter.split_documents(excel_contents)

print(len(excel_chunks))

27


#### 1-2. PDF 파일 처리

PDF 파일은 PyPDFLoader 라이브러리를 이용해 쉽게 처리할 수 있습니다!

In [9]:
# PDF
from langchain_community.document_loaders import PyPDFLoader


pdf_contents = PyPDFLoader("data/키키테크_AI솔루션_제품카탈로그.pdf").load()

print(len(pdf_contents)) # 12 = PDF 페이지 수
print(pdf_contents[1].page_content) # 각 인덱스의 page_content에는 pdf의 텍스트가 담김.

incorrect startxref pointer(1)
parsing for Object Streams


12
1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두고 판교, 부산에 지사를 운영 중이며, AI 연구개발 인력
120명을 포함한 총 350명의 임직원이 고객 가치 창출을 위해 노력하고 있습니다. 2024년
매출액 480억원을 기록하며 3년 연속 30% 이상의 성장세를 이어가고 있습니다.
주요 경쟁력
 LLM 파인튜닝 및 RAG 기술 내재화 — 자체 AI 연구팀 보유
 온프레미스/클라우드/하이브리드 배포 모두 지원
 금융위원회 마이데이터 사업자 허가, ISO 27001 인증 보유
 24시간 365일 SLA 기반 기술 지원 체계
 도입 후 6개월 이내 ROI 달성 보장 (조건부)
1.2 솔루션 포트폴리오 전체 구성
 제품명
 카테고리
 주요 대상
 핵심 기술
 TalkBridge Enterprise
 AI 챗봇
 CS, 내부 지식관리
 LLM + RAG
 DataSight AI
 데이터 분석
 경영진, 데이터팀
 NL2SQL + 시각화
 DocuMind
 문서 처리
 법무, 행정, 금융
 OCR + LLM
 PredictCore
 예측 분석
 제조, 물류, 금융
 ML + 이상감지
 AgentFlow
 에이전트 플랫폼
 IT/DevOps, 연구개발
 Multi-Agent


결과를 보면 pdf 문서가 총 12장이므로, 문서도 12장이 생긴걸 볼 수 있습니다.

이제 이 문서의 텍스트를 더 작게 분할해 보겠습니다. 분할에는 `RecursiveCharacterTextSplitter`를 사용할 수 있습니다.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size = 한 청크의 최대 토큰 수 / chunk_overlap = 겹치는 영역의 토큰 수
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pdf_chunks = splitter.split_documents(pdf_contents)

print(len(pdf_chunks))

24


#### 1-3. 워드 파일 처리

워드 파일도 Docx2txtLoader 라는 라이브러리를 사용하면 쉽게 처리할 수 있습니다.

PDF와 마찬가지로 텍스트로 변환한 뒤에 더 작은 텍스트로 분할하면 끝입니다.

In [11]:
# Word
from langchain_community.document_loaders import Docx2txtLoader
word_contents = Docx2txtLoader("data/키키테크_사내규정_행동강령.docx").load()

print(len(word_contents))
print(word_contents[0].page_content)

1
(주)키키테크

임직원 행동강령 및 사내 규정 안내서

Ver. 2025.01 | 인사팀 발행

모든 임직원은 본 안내서를 숙지하고 준수하여야 합니다.

목차

1장. 회사 소개 및 핵심 가치

2장. 임직원 행동강령

3장. 근무 규정

4장. 복리후생 안내

5장. IT 보안 정책

6장. 성희롱·괴롭힘 예방 정책

7장. 비상 연락 및 안전 규정




1장. 회사 소개 및 핵심 가치

1.1 회사 개요

(주)키키테크는 2015년 설립된 AI 솔루션 전문기업으로, 기업용 데이터 분석 플랫폼과 AI 에이전트 개발 서비스를 제공합니다. 현재 서울 강남구 테헤란로 본사를 비롯하여 판교, 부산 지사를 운영하고 있으며, 임직원 수는 약 350명입니다.

회사는 '기술로 연결하는 더 나은 내일'이라는 비전 아래, 고객사의 디지털 전환을 돕는 다양한 AI 솔루션을 연구·개발하고 있습니다.

1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없이 탐구하고 도전합니다.

신뢰 (Trust): 고객, 동료, 파트너와의 관계에서 언제나 투명하고 책임감 있게 행동합니다.

협력 (Collaboration): 다양한 배경과 역량을 가진 구성원들이 함께 더 큰 성과를 만들어냅니다.

성장 (Growth): 개인과 조직이 함께 성장하는 문화를 지향합니다.

다양성 (Diversity): 다양한 시각과 경험을 존중하며 포용적인 조직 문화를 만들어 갑니다.

1.3 조직 구조

키키테크는 다음과 같은 본부 체계로 운영됩니다:

본부

주요 팀

주요 업무

AI 연구개발본부

ML팀, LLM팀, 데이터팀

AI 모델 연구 및 제품 개발

플랫폼개발본부

백엔드팀, 프론트엔드팀, DevOps팀

서비스 플랫폼 구축 및 운영

사업본부

영업팀, 마케팅팀, 파트너십팀

고객 발굴 및 사업 확장

경영지원본부

인사팀, 재무팀, 법무팀

내부 운영 및 지원




2장. 임직원 행동강령

2.1 기본 윤리 원칙

모든 임직원은 

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF/Word처럼 긴 문서는 splitting 필요
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
word_chunks = splitter.split_documents(word_contents)

print(len(word_chunks))

13


#### 1-4. ppt 파일 처리

ppt 파일은 엑셀과 유사하게 직접 처리해서 Document 형태로 변환해 줄 필요가 있습니다.

Presentation 라이브러리를 활용해서 데이터를 로드해보겠습니다.

In [13]:
from pptx import Presentation
from langchain_core.documents import Document

def parse_pptx(file_path: str) -> list[Document]:
    # ppt 파일 로드
    prs = Presentation(file_path)
    docs = []
    # 각 슬라이드를 돌며...
    for slide_num, slide in enumerate(prs.slides, 1):
        texts = []
        # 슬라이드의 모든 요소를 살펴보며 텍스트가 있으면 texts 리스트에 추가.
        for shape in slide.shapes:
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    if para.text.strip():
                        texts.append(para.text.strip())
        if texts:
            docs.append(Document(
                page_content="\n".join(texts),
                metadata={
                    "source": file_path,
                    "slide": slide_num,
                }
            ))
    return docs

ppt_contents = parse_pptx("data/키키테크_회사소개.pptx")

print(len(ppt_contents))
print(ppt_contents[0].page_content)

5
(주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출


각 슬라이드마다 문서가 하나씩 생성된 것을 볼 수 있습니다. ppt 특성상 각 슬라이드에 텍스트가 많은 편이 아니기 때문에 별도로 splitting 과정을 거치진 않겠습니다.

In [19]:
# 지금까지 처리한 문서들을 모두 합침.
total_contents = excel_contents + word_chunks + pdf_chunks + ppt_contents

### 2. 텍스트 벡터로 변환

이제 llm이 이해할 수 있는 숫자 벡터 형태로 텍스트를 변환해 줘야 합니다.

벡터로 변환하는 방법에는 2가지가 있습니다.

1. BM25 활용해서 sparse vector 생성하기

In [22]:
from langchain_community.retrievers import BM25Retriever

# BM25 sparse retriever 생성 (rank_bm25 패키지 필요: pip install rank-bm25)
bm25_retriever = BM25Retriever.from_documents(total_contents)
bm25_retriever.k = 3  # 상위 3개 문서 반환

# 테스트
results = bm25_retriever.invoke("키키테크의 주요 수익모델은 뭐야?")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source', '엑셀 데이터')}")
    print(f"내용: {doc.page_content[:200]}")


--- Result 1 ---
출처: data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 9. 가격 정책 및 라이선스
키키테크의 모든 제품은 구독형(SaaS)과 영구 라이선스(온프레미스) 두 가지 방식으로
제공됩니다. 아래 가격은 기준 가격이며, 도입 규모, 계약 기간, 번들 구성에 따라 협의
할인이 가능합니다.
TalkBridge Enterprise 가격표
 플랜
 월 구독료
 포함 사용자
 문서 한도
 Starter
 월 200만원
 50명

--- Result 2 ---
출처: data/키키테크_사내규정_행동강령.docx
내용: 1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없이 탐구하고 도전합니다.

신뢰 (Trust): 고객, 동료, 파트너와의 관계에서 언제나 투명하고 책임감 있게 행동합니다.

협력 (Collaboration): 다양한 배경과 역량을 가진 구성원들이 함께 더 큰 성과를 만들어냅니다.

성장 (Grow

--- Result 3 ---
출처: data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 3. DataSight AI — 지능형 데이터 분석 플랫폼
3.1 제품 개요
DataSight AI는 비전문가도 자연어로 데이터를 분석하고 시각화할 수 있는 지능형 데이터 분석
플랫폼입니다. '지난 분기 지역별 매출 상위 5개 제품을 보여줘'와 같은 자연어 질의만으로
복잡한 SQL 쿼리와 시각화 차트를 자동으로 생성합니다.
3.2 주요 기능
 NL2SQL


2. OpenAIEmbedding과 FAISS 활용해서 dense vector 생성하기

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

import os
import numpy as np
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

# 사전 학습된 임베딩 모델 로드
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [24]:
# 벡터로 변환
vectorstore = FAISS.from_documents(total_contents, OpenAIEmbeddings())

vectors = np.zeros((vectorstore.index.ntotal, vectorstore.index.d), dtype=np.float32)
vectorstore.index.reconstruct_n(0, vectorstore.index.ntotal, vectors)

print(vectors.shape)
print(vectors[0][:10])

(46, 1536)
[-0.02126501 -0.03240383  0.00808097 -0.02995222 -0.01959952  0.04143746
 -0.01450977  0.01518929  0.00484325 -0.00788777]


총 124개의 문서가 생성되었으며, 각 문서는 1536개의 숫자로 이뤄진 벡터로 변환되었습니다.

이를 이용해서 질문과 가장 연관성 있는 상위 3개의 문서를 로드해 보겠습니다.

In [25]:
results = vectorstore.similarity_search("키키테크의 주요 수익모델은 뭐야?", k=3)
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source')} ({doc.metadata.get('file_type')}, {doc.metadata.get('chunk_id')})")
    print(f"내용: {doc.page_content[:200]}")


--- Result 1 ---
출처: data/키키테크_AI솔루션_제품카탈로그.pdf (None, None)
내용: 1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두

--- Result 2 ---
출처: data/키키테크_회사소개.pptx (None, None)
내용: (주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출

--- Result 3 ---
출처: data/키키테크_사내규정_행동강령.docx (None, None)
내용: (주)키키테크

임직원 행동강령 및 사내 규정 안내서

Ver. 2025.01 | 인사팀 발행

모든 임직원은 본 안내서를 숙지하고 준수하여야 합니다.

목차

1장. 회사 소개 및 핵심 가치

2장. 임직원 행동강령

3장. 근무 규정

4장. 복리후생 안내

5장. IT 보안 정책

6장. 성희롱·괴롭힘 예방 정책

7장. 비상 연락 및 안전 규정





이렇게 생성된 텍스트 벡터는 이 노트북이 종료되면 사라지게 됩니다. 사라지지 않도록 로컬에 벡터 데이터를 저장해 봅시다.

In [26]:
vectorstore.save_local("faiss_index")

저장된 벡터는 아래와 같이 불러올 수 있습니다.

In [27]:
vectorstore = FAISS.load_local(
    "faiss_index",
    OpenAIEmbeddings(),
    allow_dangerous_deserialization=True
)

### 3. sparse vector와 dense vector 섞어서 사용하기

일반적으로 retriever에는 sparse retriever와 dense retriever를 일정 비율로 섞어서 관련 문서를 찾아옵니다.

한 번 구현해 봅시다

#### 3-1. EnsembleRetriever — Sparse + Dense 결합

`EnsembleRetriever`는 두 retriever의 결과를 **RRF(Reciprocal Rank Fusion)** 알고리즘으로 합산합니다.

| 방식 | 장점 | 단점 |
|------|------|------|
| **Sparse (BM25)** | 고유명사·숫자 등 키워드 정확 매칭 | 의미 유사도 파악 불가 |
| **Dense (FAISS)** | 문맥·의미 기반 유사도 검색 | 정확한 키워드 매칭 약함 |
| **Hybrid (Ensemble)** | 두 방식의 장점 보완 | — |

`weights`로 각 retriever의 기여 비율을 조절할 수 있으며, 합이 1이 되어야 합니다.

In [33]:
from langchain_classic.retrievers import EnsembleRetriever

# FAISS dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Sparse(BM25) 40% + Dense(FAISS) 60% 비율로 혼합
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]
)

# 테스트: 동일한 질문으로 하이브리드 검색 결과 확인
results = ensemble_retriever.invoke("키키테크의 주요 수익모델은 뭐야?")
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"출처: {doc.metadata.get('source', '엑셀 데이터')}")
    print(f"내용: {doc.page_content[:200]}")


--- Result 1 ---
출처: data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두

--- Result 2 ---
출처: data/키키테크_회사소개.pptx
내용: (주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출

--- Result 3 ---
출처: data/키키테크_사내규정_행동강령.docx
내용: (주)키키테크

임직원 행동강령 및 사내 규정 안내서

Ver. 2025.01 | 인사팀 발행

모든 임직원은 본 안내서를 숙지하고 준수하여야 합니다.

목차

1장. 회사 소개 및 핵심 가치

2장. 임직원 행동강령

3장. 근무 규정

4장. 복리후생 안내

5장. IT 보안 정책

6장. 성희롱·괴롭힘 예방 정책

7장. 비상 연락 및 안전 규정




--- Result 4 ---
출처: data/키키테크_AI솔루션_제품카탈로그.pdf
내용: 9. 가격 정책 및 라이선스
키키테크의 모든 제품은 구독형(SaaS)과 영구 라이선스(온프레미스) 두 가지 방식으로
제공됩니다. 아래 가격은 기준 가격이며, 도입 규모, 계약 기간, 번들 구성에 따라 협의
할인이 가능합니다.
TalkBridge Enterprise 가격표
 플랜
 월 구독료
 포함 사용자
 문서 한도
 Starter
 월 200만원
 50명

--- Result 5 ---
출처: data/키키테크_사내규정_행동강령.docx
내용: 1.2 핵심 가치 (Core Values)

혁신 (Innovation): 새로운 기술과 아이디어를 두려움 없이 탐구하고 도전합니다

### 4. RAG tool로 변환하기

앞서 살펴본 rag 방식을 함수로 묶어서 tool로 변환하고 agent에 추가해서 rag를 구현해 보겠습니다!

In [34]:
from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """키키테크 사내 문서(임직원 현황, 제품 카탈로그, 사내 규정, 회사 소개)에서 관련 내용을 검색합니다."""
    docs = ensemble_retriever.invoke(query)
    if not docs:
        return "관련 문서를 찾을 수 없습니다."
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

print(search_documents.invoke({"query": "키키테크의 주요 수입모델은?"}))

(주)키키테크
회사 소개
기술로 연결하는 더 나은 내일
2025
2015
설립연도
350+
임직원 수
480억
2024 매출

---

1. 회사 개요 및 솔루션 포트폴리오
1.1 (주)키키테크 소개
(주)키키테크는 2015년 설립된 국내 대표 AI 솔루션 전문기업입니다. 창립 이래 기업의 디지털
전환을 선도하는 혁신적인 AI 기술을 연구·개발해 왔으며, 현재 금융, 제조, 유통, 의료,
법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있습니다.
서울 강남구 테헤란로에 본사를 두고 판교, 부산에 지사를 운영 중이며, AI 연구개발 인력
120명을 포함한 총 350명의 임직원이 고객 가치 창출을 위해 노력하고 있습니다. 2024년
매출액 480억원을 기록하며 3년 연속 30% 이상의 성장세를 이어가고 있습니다.
주요 경쟁력
 LLM 파인튜닝 및 RAG 기술 내재화 — 자체 AI 연구팀 보유
 온프레미스/클라우드/하이브리드 배포 모두 지원
 금융위원회 마이데이터 사업자 허가, ISO 27001 인증 보유
 24시간 365일 SLA 기반 기술 지원 체계

---

회사 개요
(주)키키테크는 2015년 설립된 AI 솔루션 전문기업으로, 기업의 디지털 전환을 선도하는 혁신적인 AI 기술을 연구·개발하고 있습니다. 금융, 제조, 유통, 의료, 법률 등 다양한 산업군의 350여 개 기업 고객을 보유하고 있으며, 서울 강남구 본사를 비롯해 판교·부산 지사를 운영 중입니다.
혁
혁신
새로운 기술과
아이디어 탐구
신
신뢰
투명하고 책임감
있는 파트너십
협
협력
다양한 역량의
구성원이 함께
성
성장
개인과 조직이
함께 성장
다
다양성
포용적 조직
문화 지향
비전: 기술로 연결하는 더 나은 내일

---

9. 가격 정책 및 라이선스
키키테크의 모든 제품은 구독형(SaaS)과 영구 라이선스(온프레미스) 두 가지 방식으로
제공됩니다. 아래 가격은 기준 가격이며, 도입 규모, 계약 기간, 번들 구성에 따라 협의
할인이 가능합니다.
TalkBridge Enterpris

### 4. 에이전트에 tool 추가하기

이제 에이전트에 위에서 구현한 tool을 추가해 보도록 하겠습니다!

In [35]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# 모델 이름
model_name = "gpt-5.4-mini"

# LLM 정의
model = init_chat_model(
    model=model_name,
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.7,
)

agent = create_agent(
    model,
    tools=[search_documents],
    system_prompt="너는 키키테크의 사내 Q&A 챗봇이다. 기업과 관련된 정보에 대해 문서를 꼭 확인하고 답을 생성해야 한다. 문서에서 확인할 수 없는 질문에는 대답하지 말 것."
)

response = agent.invoke(
    {"messages": {"role": "user", "content": "키키테크의 주요 수익모델은 뭐야?"}}
)

print(response['messages'][-1].content)

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


문서 기준으로 보면, 키키테크의 주요 수익모델은 **엔터프라이즈 AI 솔루션 판매 및 도입 관련 수익**입니다.

확인된 내용:
- **AI 솔루션 제품 카탈로그**에 제품별 **가격 정책**과 **도입 문의/컨설팅 안내**가 포함되어 있음
- 회사 소개/사내 규정에는 키키테크가 **기업용 데이터 분석 플랫폼과 AI 에이전트 개발 서비스**를 제공한다고 나와 있음
- 주요 제품군으로는:
  - **TalkBridge Enterprise**: 기업용 AI 챗봇 플랫폼
  - **DataSight AI**: 지능형 데이터 분석 플랫폼
  - **DocuMind**: AI 문서 처리 자동화 솔루션
  - **PredictCore**: 예측 분석 및 이상감지 엔진
  - **AgentFlow**: 멀티 에이전트 워크플로우 플랫폼
- 또한 **기술 지원 및 유지보수 정책**, **가격 정책 및 라이선스** 항목이 있어, 라이선스/유지보수/컨설팅이 수익원에 포함되는 것으로 해석됩니다.

다만 문서에 **“주요 수익모델”을 명시적으로 한 문장으로 정의한 부분은 없어서**, 위처럼 문서에서 확인되는 범위까지만 말씀드릴 수 있습니다.
